# 03 — Model improvements at 15 s (E04–E06)

This notebook **reads** results; it trains nothing. Every run was made by
`scripts/run_experiment.py` from a committed config (`configs/experiments/E0[4-6]_*.yaml`, written
by `scripts/make_configs.py`), logged to MLflow and exported to `reports/history.csv` by
`scripts/export_history.py`. The same tables, in text, are in `reports/phase3_tables.md`
(`scripts/results_tables.py --phase 3`). This notebook is written by `scripts/make_notebooks.py`.

Protocol (data v1): stratified 70/15/15 split with seed 42, 5-fold CV on the training set, fit
on the training set, score on the validation set; the test set is not touched. Each experiment
changes one thing against the current best and runs first at 15 s for all four tasks. A model
is rerun at 30, 60 and 120 s only if its 15 s validation macro F1 beats the task's current best
by at least 0.03 (the winner rule, decided 28 Oct).

- **E04:** KNN with a signed log, sign(x)·log1p(|x|), then StandardScaler or RobustScaler.
- **E05:** Random Forest, 300 trees, default settings.
- **E06:** XGBoost and LightGBM, default settings.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from blindspy import results as R
from blindspy.data import CORE_TASKS, LABELS, ROOT
from blindspy.tracking import load_yaml

pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_colwidth", 80)
FIG = ROOT / "reports" / "figures"
RUNNER = load_yaml(ROOT / "configs" / "base.yaml")["official_runner"]
h = R.load_history()
s = R.selection_runs(h)


def verdict_table(v: dict, prefixes: tuple) -> pd.DataFrame:
    rows = [{"check": k, "supported": x.get("supported"),
             "details": json.dumps({a: b for a, b in x.items() if a != "supported"},
                                   default=lambda o: o.item() if hasattr(o, "item") else str(o))}
            for k, x in v.items() if k.startswith(prefixes)]  # fmt: skip
    return pd.DataFrame(rows)


EXPS = ["E04", "E05", "E06"]
p3 = s[s["tags.experiment_id"].isin(EXPS)]
print(
    len(p3),
    "runs of E04-E06 (15 s screen and winner reruns); data versions:",
    sorted(set(p3["tags.data_version"])),
    "; commits:",
    sorted(set(p3["tags.git_commit"].str[:7])),
)

## 1. Screen at 15 s: each experiment against the best run before it

In [ ]:
for exp in EXPS:
    print(exp)
    display(R.screen(s, exp))
print("Current best at 15 s after E06")
display(R.current_best(s, "E06"))

**Findings (screen at 15 s, validation, data v1).**

- **E04:** the signed log with StandardScaler lifts KNN above the E03 KNN in all four tasks
  (+0.0801 to +0.1040) and beats the RobustScaler variant in all four (by 0.0118–0.0224). It
  also passes the E02 tree everywhere (+0.0059 to +0.0376), so it becomes the current best in
  every task. Only A2-NonVPN (+0.0376) clears the 0.03 winner margin; A2-VPN misses it by 0.0004
  (+0.0296).
- **E05:** the 300-tree Random Forest is the current best in all four tasks (+0.0135 to +0.0503
  over the E04 KNN); only B (+0.0503) is a winner.
- **E06:** default XGBoost is the current best in all four tasks (+0.0067 to +0.0156 over the
  forest); LightGBM is 0.0032–0.0102 below XGBoost in every task and below the forest in A1
  (−0.0022). No winner.
- Current best after E06: XGBoost with default settings everywhere (A1 0.9271, A2-VPN 0.8758,
  A2-NonVPN 0.9015, B 0.8303).

## 2. Model comparison at 15 s: validation macro F1 with CV mean ± std

In [ ]:
MODELS = [
    "E02-tree",
    "E03-knn",
    "E03-logreg",
    "E03-svm_rbf",
    "E04-knn_log_standard",
    "E04-knn_log_robust",
    "E05-rf",
    "E06-xgb",
    "E06-lgbm",
]
COLORS = ["#bbbbbb"] * 4 + ["#ff7f0e"] * 2 + ["#2ca02c"] + ["#1f77b4"] * 2
q = s[(s["params.timeout"] == 15) & s["label"].isin(MODELS)]
fig, axes = plt.subplots(1, 4, figsize=(16, 4.4), sharey=True)
for ax, task in zip(axes, CORE_TASKS, strict=True):
    t = q[q["params.task"] == task].set_index("label").loc[MODELS]
    x = np.arange(len(MODELS))
    ax.bar(x, t["metrics.val_f1_macro"], color=COLORS)
    ax.errorbar(
        x,
        t["metrics.cv_f1_macro_mean"],
        yerr=t["metrics.cv_f1_macro_std"],
        fmt="o",
        color="black",
        ms=3,
        capsize=2,
    )
    ax.set_xticks(x, MODELS, rotation=60, ha="right", fontsize=8)
    ax.set_title(f"{task}, 15 s")
    ax.set_ylim(0, 1)
axes[0].set_ylabel("macro F1\n(bar: validation; dot: 5-fold CV mean +- std)")
fig.suptitle("E02-E03 baselines (grey) and E04-E06 at 15 s, data v1")
fig.tight_layout()
fig.savefig(FIG / "phase3_model_comparison_E04-E06.png", dpi=120)

Winner reruns at every timeout (the winner rule fired for these two only):

In [ ]:
display(R.timeout_table(s, "E04-knn_log_standard", "A2-NonVPN"))
display(R.timeout_table(s, "E05-rf", "B"))

**Findings.**

- From the E04 KNN to XGBoost the improved models lie close together: their spread per task is
  0.034 (A1: 0.8929–0.9271), 0.039 (A2-VPN), 0.048 (A2-NonVPN) and 0.088 (B), against
  0.31–0.46 between the E02 tree and the weakest E03 baseline.
- Cross-validation (dots) and validation (bars) differ by −0.021 to +0.024 (CV − validation).
  In A2-NonVPN CV is higher than validation for every E04–E06 model (+0.011 to +0.024); in the
  other tasks the difference is between −0.021 and +0.015. E14 later shows why: the official
  A2-NonVPN validation rows are harder than typical.
- Winner reruns: the signed-log KNN in A2-NonVPN scores 0.8523 / 0.8590 / 0.8491 at 30 / 60 /
  120 s and stays the best run at each timeout (by 0.0240 / 0.0344 / 0.0098); the forest in B
  scores 0.8114 / 0.7771 / 0.7835 and leads the tree by 0.0638 / 0.0713 / 0.0677. Both score
  less at 60 and 120 s than at 15 s. The class mix differs between timeouts and the 60 s files
  hold differently shaped VOIP and FT flows (E00 addendum), so this is not read as a timeout
  effect.

## 3. Per-class F1 at 15 s (validation)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(17, 10))
for ax, task in zip(axes.flat, CORE_TASKS, strict=True):
    t = q[q["params.task"] == task].set_index("label").loc[MODELS]
    classes = LABELS[task]
    m = t[[f"metrics.val_f1.{c}" for c in classes]].to_numpy(dtype=float)
    ax.imshow(m, cmap="viridis", vmin=0, vmax=1, aspect="auto")
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            ax.text(
                j,
                i,
                f"{m[i, j]:.2f}",
                ha="center",
                va="center",
                fontsize=7,
                color="white" if m[i, j] < 0.6 else "black",
            )
    ax.set_xticks(range(len(classes)), classes, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(MODELS)), MODELS, fontsize=8)
    ax.set_title(f"{task}, 15 s: validation F1 per class")
fig.tight_layout()
fig.savefig(FIG / "per_class_f1_E04-E06.png", dpi=120)

**Findings (values rounded to 2 decimals in the chart).**

- VoIP is the easiest traffic type for every E04–E06 model (F1 0.96–1.00 for VOIP and
  VPN-VOIP).
- The hardest classes are the mail and chat classes and non-VPN streaming: in B, MAIL
  0.56–0.72, VPN-MAIL 0.58–0.66 and VPN-CHAT 0.56–0.72; in A2-VPN, VPN-CHAT 0.63–0.72; in
  A2-NonVPN, STREAMING 0.73–0.82.
- The gains over the E02 tree come mostly from these weak classes, e.g. XGBoost in B: CHAT
  0.70 → 0.80, VPN-CHAT 0.58 → 0.72, MAIL 0.61 → 0.72; in A2-NonVPN: CHAT 0.76 → 0.88, FT
  0.76 → 0.88.
- The signed log repairs the classes the E03 KNN got worst: A2-NonVPN CHAT 0.63 → 0.83 and
  STREAMING 0.63 → 0.74, B BROWSING 0.64 → 0.83.

## 4. Cost: training and prediction time, model size, memory

In [ ]:
for exp in EXPS:
    print(exp)
    display(R.run_details(s, exp))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 7.5), sharey="row")
for c, task in enumerate(CORE_TASKS):
    t = q[q["params.task"] == task].set_index("label").loc[MODELS]
    for r, (col, xlabel) in enumerate(
        [
            ("metrics.train_time_s", "training time (s, log)"),
            ("metrics.predict_ms_per_1000", "prediction time per 1,000 flows (ms, log)"),
        ]
    ):
        ax = axes[r][c]
        for (label, row), color in zip(t.iterrows(), COLORS, strict=True):
            ax.scatter(row[col], row["metrics.val_f1_macro"], color=color, s=30)
            ax.annotate(
                label.split("-", 1)[1],
                (row[col], row["metrics.val_f1_macro"]),
                fontsize=7,
                xytext=(3, 2),
                textcoords="offset points",
            )
        ax.set_xscale("log")
        ax.set_xlabel(xlabel)
        ax.set_title(f"{task}, 15 s")
    axes[0][0].set_ylabel("validation macro F1")
    axes[1][0].set_ylabel("validation macro F1")
fig.tight_layout()
fig.savefig(FIG / "cost_E04-E06.png", dpi=120)

**Findings (15 s runs; one thread per model, the official machine).**

- Training: the KNN only stores the rows (0.014–0.033 s); XGBoost takes 0.256–2.340 s,
  LightGBM 0.191–2.015 s and the forest 2.038–6.059 s.
- Prediction per 1,000 flows: XGBoost 1.13–11.76 ms, KNN 7.75–11.50 ms, the forest
  21.91–27.28 ms, LightGBM 5.12–98.28 ms. LightGBM grows one tree per class per round (100 trees
  in A1, 1,400 in B), so its prediction time grows most with the number of classes.
- Model files (compressed): the forest 5.31–22.44 MiB (171.86 MiB uncompressed for B), XGBoost
  0.12–1.06 MiB, LightGBM 0.14–2.01 MiB. The forest's fit needs 33.1–187.9 MiB in a fresh process
  (child RSS), which tracemalloc does not see (1.4–2.5 MiB).
- So default XGBoost is the best E04–E06 model and also one of the cheapest to store and to run.

## 5. Threshold checks (pre-stated in E04–E06)

In [ ]:
verdict_table(R.phase3_verdicts(s, RUNNER), ("E04", "E05", "E06"))

**Summary of the pre-stated thresholds:** all six are supported
(E04a, E04b, E05a, E05b, E06a, E06b). The current best after E06 is default XGBoost in every
task.